# Runoff Analysis — Observed vs. Simulated (J2000 Output)

**Purpose:** Loads J2000 model output (`TimeLoop.dat`) and produces two
multi-panel discharge comparison plots — one for the calibration/validation
split, one for a freely defined date window.

**What it does:**
- Parses `TimeLoop.dat` (or fallback `.dat.txt`)
- Plots observed (`catchmentObsRunoff`) vs. simulated (`catchmentSimRunoff_qm`)
- Adds German month labels via locale setting
- Exports two PNG figures

**Input:** `TimeLoop_1989.dat`  
**Output:** Two PNG discharge comparison plots

---

# Abflussanalyse: Beobachtet vs. simuliert

Dieses Notebook lädt `TimeLoop_1989.dat` (Fallback `TimeLoop_1989.dat.txt`) und erzeugt zwei PNG-Diagramme: (1) Kalibrierung/Validierung, (2) frei definierter Ausschnitt.


In [ ]:

# %%
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, HPacker
import numpy as np
import locale
try:
    # für deutsche Monatsnamen; fällt sonst automatisch auf System-Standard (englisch) zurück
    locale.setlocale(locale.LC_TIME, 'de_DE.UTF-8')
except locale.Error:
    pass

# -------------------------
# Parameter
# -------------------------

file_candidates = [Path(base_name), Path(base_name + ".txt")]
initial_start = "2011-11-01 00:00"
initial_end   = "2012-10-31 23:00"
calib_start   = "2012-11-01 00:00"
calib_end     = "2017-10-31 23:00"
valid_start   = "2017-11-01 00:00"
valid_end     = "2022-10-31 23:00"
cutout_start  = "2013-05-15 00:00"
cutout_end    = "2013-07-15 23:00"
COL_ID  = "ID"
COL_QOBS = "catchmentObsRunoff"
COL_QSIM = "catchmentSimRunoff_qm"
out_dir = Path("J2k/Q_comparison"); out_dir.mkdir(exist_ok=True)

# -------------------------
# Utilities
# -------------------------


def _extract_file_label_and_id(file_path: Path):
    """
    Liefert (file_label, run_id), z. B. ("TimeLoop_1989", "1989").
    Funktioniert für 'TimeLoop_1989.dat' und 'TimeLoop_1989.dat.txt'.
    """
    label = file_path.stem                   # bei *.dat.txt -> 'TimeLoop_1989.dat'
    if label.endswith('.dat'):               # Sonderfall bereinigen
        label = label[:-4]                   # -> 'TimeLoop_1989'
    m = re.search(r'(\d+)(?!.*\d)', label)   # letzte Ziffernfolge
    run_id = m.group(1) if m else "NA"
    return label, run_id

def _safe_slug(s: str):
    """Für Dateinamen: Sonderzeichen neutralisieren."""
    return re.sub(r'[^A-Za-z0-9_\-]+', '_', s).strip('_')

def style_time_axis_full(ax):
    """
    Für lange Zeiträume (~10 Jahre):
    - Major: Jahre (beschriftet)
    - Minor: Monate (nur Ticks, unbeschriftet)
    """
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    # Tick-Längen und Gitter leicht unterscheiden
    ax.tick_params(axis='x', which='major', length=7)
    ax.tick_params(axis='x', which='minor', length=3)
    # feines vertikales Raster auf Monatsbasis
    ax.grid(True, which='minor', axis='x', alpha=0.08)


def style_time_axis_cutout(ax, start, end):
    """
    Dynamische Achse für den Ausschnitt:
    - > 120 Tage: Monate
    - 22–120 Tage: Wochen (Mo) als Major, Tage als Minor
    - 4–21 Tage: Tage als Major, 6h als Minor
    - <= 3 Tage: 3h als Major, 1h als Minor
    """
    span_days = (end - start).total_seconds() / 86400.0

    if span_days > 120:
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))   # z. B. Mär 2026
        ax.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    elif span_days > 21:
        ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%b'))   # z. B. 04.Mär
        ax.xaxis.set_minor_locator(mdates.DayLocator())
    elif span_days > 3:
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%y'))
        ax.xaxis.set_minor_locator(mdates.HourLocator(interval=6))
    else:
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m. %H:%M'))
        ax.xaxis.set_minor_locator(mdates.HourLocator(interval=1))

    ax.tick_params(axis='x', which='major', length=7)
    ax.tick_params(axis='x', which='minor', length=3)
    ax.grid(True, which='both', axis='x', alpha=0.15)

def _deduplicate(names):
    seen, out = {}, []
    for n in names:
        if n in seen:
            seen[n] += 1; out.append(f"{n}.{seen[n]}")
        else:
            seen[n] = 0; out.append(n)
    return out


def read_timeloop(path_candidates):
    file_path = next((p for p in path_candidates if p.exists()), None)
    if file_path is None:
        raise FileNotFoundError("Keine der Dateien existiert: " + ", ".join(map(str, path_candidates)))

    lines = file_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    # @attributes lokalisieren
    try:
        idx_attr = next(i for i, s in enumerate(lines) if s.strip().lower() == "@attributes")
    except StopIteration:
        import io
        df = pd.read_csv(file_path, sep="\t", header=None, engine="python")
        cols = [COL_ID] + [f"var{i}" for i in range(1, df.shape[1])]
        df.columns = cols
        return df, file_path

    j = idx_attr + 1
    while j < len(lines) and not lines[j].strip():
        j += 1
    raw_cols = re.split(r"\t+", lines[j].rstrip("\t"))
    cols = _deduplicate([c.strip() for c in raw_cols if c.strip()])

    try:
        idx_start = next(i for i, s in enumerate(lines) if s.strip().lower() == "@start")
    except StopIteration:
        try:
            idx_data = next(i for i, s in enumerate(lines) if s.strip().lower() == "@data")
            idx_start = idx_data
        except StopIteration:
            pattern = re.compile(r"^\s*\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}\b")
            idx_start = next(i for i, s in enumerate(lines) if pattern.search(s)) - 1

    import io
    data_text = "\n".join(lines[idx_start + 1:])
    df = pd.read_csv(io.StringIO(data_text), sep="\t", header=None, names=cols, usecols=range(len(cols)), engine="python")
    return df, file_path


def validate_periods(df, init_s, init_e, cal_s, cal_e, val_s, val_e):
    if not pd.api.types.is_datetime64_any_dtype(df[COL_ID]):
        raise TypeError("Spalte 'ID' konnte nicht als Datum/Zeit interpretiert werden.")
    df.sort_values(COL_ID, inplace=True)
    df.drop_duplicates(subset=[COL_ID], inplace=True)
    if not (init_s <= init_e < cal_s <= cal_e < val_s <= val_e):
        raise ValueError("Zeiträume müssen geordnet sein: Initialisierung < Kalibrierung < Validierung (von-bis).")


def plot_main(df, cal_s, cal_e, val_s, val_e, out_dir, file_label, run_id):
    # Daten für Gesamtdarstellung (calib..valid)
    msk_all = (df[COL_ID] >= cal_s) & (df[COL_ID] <= val_e)
    d = df.loc[msk_all, [COL_ID, COL_QOBS, COL_QSIM]].copy()

    # Daten für Metriken (getrennt nach Zeiträumen)
    d_cal = df.loc[(df[COL_ID] >= cal_s) & (df[COL_ID] <= cal_e), [COL_QOBS, COL_QSIM]].copy()
    d_val = df.loc[(df[COL_ID] >= val_s) & (df[COL_ID] <= val_e), [COL_QOBS, COL_QSIM]].copy()

    # Kennzahlen berechnen
    m_cal = compute_metrics(d_cal[COL_QOBS].values, d_cal[COL_QSIM].values)
    m_val = compute_metrics(d_val[COL_QOBS].values, d_val[COL_QSIM].values)

    # Plot
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(d[COL_ID], d[COL_QOBS], color="black",     linewidth=1.0, label="Q beobachtet")
    ax.plot(d[COL_ID], d[COL_QSIM], color="royalblue", linewidth=1.0, label="Q simuliert")

    # Zeiträume visuell markieren
    ax.axvspan(cal_s, cal_e, color="royalblue", alpha=0.08, label="Kalibrierung")
    ax.axvspan(val_s, val_e, color="orange",   alpha=0.10, label="Validierung")

    # Titel/Labels
    ax.set_title(f"Beobachteter vs. simulierter Abfluss (Kalibrierung/Validierung) – {file_label}")
    ax.set_xlabel("Datum")
    ax.set_ylabel("Q [m³/s]")

    # Achsen-Styling (Jahre beschriftet, Monate als unbeschriftete Minor-Ticks)
    style_time_axis_full(ax)

    # Legende & Gitter
    ax.legend(loc="upper right")
    ax.grid(True, which="major", axis="both", alpha=0.25)

    # >>> ZWEISPALTIGE METRIKEN-BOX EINBLENDEN
    _add_metrics_box(
        ax,
        m_cal=m_cal, m_val=m_val,
        cal_s=cal_s, cal_e=cal_e, val_s=val_s, val_e=val_e,
        loc='upper left',   # Standard: oben links (Legende ist oben rechts)
        sep=24,             # Spaltenabstand (Punkt)
        fontsize=9,
        alpha=0.85
    )

    # Export
    base_slug = _safe_slug(file_label)
    out_file = out_dir / f"runoff_calib_valid_{base_slug}.png"
    fig.tight_layout()
    fig.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Gespeichert: {out_file}")


def plot_cutout(df, s, e, out_dir, file_label, run_id):
    msk = (df[COL_ID] >= s) & (df[COL_ID] <= e)
    d = df.loc[msk, [COL_ID, COL_QOBS, COL_QSIM]].copy()
    if d.empty:
        print("Hinweis: Der gewählte Ausschnitt enthält keine Daten.")
        return

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(d[COL_ID], d[COL_QOBS], color="black", linewidth=1.0, label="Q beobachtet")
    ax.plot(d[COL_ID], d[COL_QSIM], color="royalblue", linewidth=1.0, label="Q simuliert")

    # ► Titel mit Dateiname + ID
    ax.set_title(f"Beobachteter vs. simulierter Abfluss (Ausschnitt) – {file_label}")
    ax.set_xlabel("Datum")
    ax.set_ylabel("Q [m³/s]")

    # ► Dynamische Achsenlogik je nach Spanne
    style_time_axis_cutout(ax, s, e)

    ax.legend(loc="upper right")
    ax.grid(True, which="major", axis="both", alpha=0.25)

    base_slug = _safe_slug(file_label)
    out_file = out_dir / f"runoff_cutout_{s.strftime('%Y%m%d%H%M')}_{e.strftime('%Y%m%d%H%M')}_{base_slug}.png"
    fig.tight_layout()
    fig.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"Gespeichert: {out_file}")


def _fmt3(x):
    """Zahlen robust und einheitlich formatieren."""
    return f"{x:.3f}" if np.isfinite(x) else "nan"

def compute_metrics(o, s):
    """
    Berechnet NSE und KGE (Revision 2012) inkl. Komponenten r, beta, gamma.
    Erwartet 1D-Arrays/Pandas-Serien gleicher Länge. NaNs werden maskiert.
    """
    o = np.asarray(o, dtype=float)
    s = np.asarray(s, dtype=float)
    msk = np.isfinite(o) & np.isfinite(s)
    o, s = o[msk], s[msk]

    if o.size < 2:
        return dict(NSE=np.nan, KGE2012=np.nan, r=np.nan, beta=np.nan, gamma=np.nan)

    # NSE
    denom = np.sum((o - o.mean())**2)
    num   = np.sum((s - o)**2)
    nse = 1.0 - (num / denom) if denom != 0.0 else np.nan

    # Komponenten für KGE'12
    # r = Pearson-Korrelation
    std_o = np.std(o, ddof=1)
    std_s = np.std(s, ddof=1)
    if std_o == 0.0 or std_s == 0.0:
        r = np.nan
    else:
        r = np.corrcoef(o, s)[0, 1]

    mu_o = np.mean(o)
    mu_s = np.mean(s)
    beta = (mu_s / mu_o) if mu_o != 0.0 else np.nan

    cv_o = (std_o / mu_o) if mu_o != 0.0 else np.nan
    cv_s = (std_s / mu_s) if mu_s != 0.0 else np.nan
    gamma = (cv_s / cv_o) if np.isfinite(cv_s) and np.isfinite(cv_o) and cv_o != 0.0 else np.nan

    # KGE (Revision 2012)
    if np.isfinite(r) and np.isfinite(beta) and np.isfinite(gamma):
        kge12 = 1.0 - np.sqrt((r - 1.0)**2 + (beta - 1.0)**2 + (gamma - 1.0)**2)
    else:
        kge12 = np.nan

    return dict(NSE=nse, KGE2012=kge12, r=r, beta=beta, gamma=gamma)

def _add_metrics_box(ax, m_cal, m_val, cal_s, cal_e, val_s, val_e,
                     loc='upper left', sep=24, fontsize=9, alpha=0.85):
    """
    Erstellt eine zweispaltige Metriken-Box:
      - linke Spalte: Kalibrierung
      - rechte Spalte: Validierung
    loc: 'upper left' | 'upper right' | 'lower left' | 'lower right' | ...
    sep: horizontaler Abstand (in Punkten) zwischen den Spalten
    """

    left_txt = (
        f"Kalibrierung    NSE: {_fmt3(m_cal['NSE'])} \n"
        f"  KGE: {_fmt3(m_cal['KGE2012'])}    r:   {_fmt3(m_cal['r'])} \n"
        f"  β:   {_fmt3(m_cal['beta'])}    γ:   {_fmt3(m_cal['gamma'])}"
    )

    right_txt = (
        f"Validierung    NSE: {_fmt3(m_val['NSE'])} \n"
        f"  KGE: {_fmt3(m_val['KGE2012'])}   r:   {_fmt3(m_val['r'])} \n"
        f"  β:   {_fmt3(m_val['beta'])}   γ:   {_fmt3(m_val['gamma'])}"
    )

    left  = TextArea(left_txt,  textprops=dict(family='monospace', size=fontsize))
    right = TextArea(right_txt, textprops=dict(family='monospace', size=fontsize))

    # Zwei Spalten nebeneinander packen
    hbox = HPacker(children=[left, right], align="top", pad=0, sep=sep)

    anchored = AnchoredOffsetbox(
        loc=loc, child=hbox, frameon=True, borderpad=0.6,
        bbox_to_anchor=(0, 1), bbox_transform=ax.transAxes  # an Achse koppeln
    )
    # Styling der Box
    anchored.patch.set_boxstyle('round,pad=0.4')
    anchored.patch.set_alpha(alpha)
    anchored.patch.set_edgecolor('0.7')

    ax.add_artist(anchored)

def run_one(base_name: str, save_plots: bool = True):
    """
    Führt den kompletten Lauf für genau eine Eingabedatei durch:
      - Datei(en) finden (.dat / .dat.txt)
      - Einlesen, Validierung
      - Plots (groß & Ausschnitt)
      - Metriken (Kalibrierung & Validierung)
    Gibt ein Dict mit Kennzahlen und Metadaten zurück.
    """
    file_candidates = [Path(base_name), Path(base_name + ".txt")]
    df, used_path = read_timeloop(file_candidates)

    # ID-Spalte sicherstellen
    if df.columns[0] != COL_ID:
        df.rename(columns={df.columns[0]: COL_ID}, inplace=True)
    if not pd.api.types.is_datetime64_any_dtype(df[COL_ID]):
        df[COL_ID] = pd.to_datetime(df[COL_ID], format="%Y-%m-%d %H:%M", errors="coerce")

    # Dateilabel + ID
    file_label, run_id = _extract_file_label_and_id(used_path)

    # Pflichtspalten prüfen
    required = [COL_QOBS, COL_QSIM]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"Erforderliche Spalten fehlen: {missing} (Datei: {used_path}).")

    # Zeiträume
    init_s = pd.to_datetime(initial_start, format="%Y-%m-%d %H:%M")
    init_e = pd.to_datetime(initial_end,   format="%Y-%m-%d %H:%M")
    cal_s  = pd.to_datetime(calib_start,   format="%Y-%m-%d %H:%M")
    cal_e  = pd.to_datetime(calib_end,     format="%Y-%m-%d %H:%M")
    val_s  = pd.to_datetime(valid_start,   format="%Y-%m-%d %H:%M")
    val_e  = pd.to_datetime(valid_end,     format="%Y-%m-%d %H:%M")

    validate_periods(df, init_s, init_e, cal_s, cal_e, val_s, val_e)

    # --- Metriken vorbereiten (werden von plot_main ohnehin berechnet) ---
    d_cal = df.loc[(df[COL_ID] >= cal_s) & (df[COL_ID] <= cal_e), [COL_QOBS, COL_QSIM]].copy()
    d_val = df.loc[(df[COL_ID] >= val_s) & (df[COL_ID] <= val_e), [COL_QOBS, COL_QSIM]].copy()
    m_cal = compute_metrics(d_cal[COL_QOBS].values, d_cal[COL_QSIM].values)
    m_val = compute_metrics(d_val[COL_QOBS].values, d_val[COL_QSIM].values)

    # --- Plots ---
    if save_plots:
        plot_main(df, cal_s, cal_e, val_s, val_e, out_dir, file_label, run_id)
        co_s = pd.to_datetime(cutout_start, format="%Y-%m-%d %H:%M")
        co_e = pd.to_datetime(cutout_end,   format="%Y-%m-%d %H:%M")
        plot_cutout(df, co_s, co_e, out_dir, file_label, run_id)

    # --- Kompakte Rückgabe für Übersichten ---
    return {
        "file": str(used_path),
        "file_label": file_label,
        "run_id": run_id,
        # Kalibrierung
        "cal_NSE": m_cal["NSE"], "cal_KGE2012": m_cal["KGE2012"],
        "cal_r": m_cal["r"], "cal_beta": m_cal["beta"], "cal_gamma": m_cal["gamma"],
        # Validierung
        "val_NSE": m_val["NSE"], "val_KGE2012": m_val["KGE2012"],
        "val_r": m_val["r"], "val_beta": m_val["beta"], "val_gamma": m_val["gamma"],
    }
    
# -------------------------
# Hauptablauf (einzeln oder mehrere Eingabedateien)
# -------------------------

# Variante 1: Wie bisher – genau EINE Datei (nutzt oben definiertes base_name)
# inputs = [base_name]

# Variante 2: EXPLIZITE LISTE (z. B. 5 Dateien)
# inputs = [
#     "J2k/TimeLoop_1989.dat",
#     "J2k/TimeLoop_1990.dat",
#     "J2k/TimeLoop_1991.dat",
#     "J2k/TimeLoop_1992.dat",
#     "J2k/TimeLoop_1993.dat",
# ]

# Variante 3: Mit Globbing-Muster (alle passenden Dateien)
# from pathlib import Path
# inputs = sorted(str(p) for p in Path("J2k").glob("TimeLoop_*.dat"))

# → Hier die gewünschte Variante aktivieren:
inputs = [
    "J2k/TimeLoop_1935.dat",
    "J2k/TimeLoop_1947.dat",
    "J2k/TimeLoop_1973.dat",
    "J2k/TimeLoop_1978.dat",
    "J2k/TimeLoop_1989.dat",
]

summary_rows = []
for inp in inputs:
    print(f"\n=== Verarbeite: {inp} ===")
    row = run_one(inp, save_plots=True)
    summary_rows.append(row)

# Übersicht (nur sinnvoll, wenn mehrere Läufe)
if len(summary_rows) > 1:
    summary_df = pd.DataFrame(summary_rows)
    csv_path = out_dir / "metrics_summary.csv"
    summary_df.to_csv(csv_path, index=False)
    # Schöne, kurze Ausgabe in der Konsole
    with pd.option_context('display.max_columns', None, 'display.width', 160):
        print("\nGesamtübersicht (ausgewählte Spalten):")
        print(summary_df[[
            "file_label", "run_id",
            "cal_NSE", "cal_KGE2012", "cal_r", "cal_beta", "cal_gamma",
            "val_NSE", "val_KGE2012", "val_r", "val_beta", "val_gamma"
        ]])
    print(f"\nGespeichert: {csv_path}")

print("\nFertig.")
